# Short-Term Memory

In [1]:
from agents import Agent, Runner, SQLiteSession, trace
from setup import bedrock_tool

✅ AWS credentials found
✅ OpenAI credentials found
✅ EXA credentials found


In [2]:
nutrition_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful assistant comparing how healthy different foods are.
    If you answer, give a list of how healthy the foods are with a score from 1 to 10. Order by: healtiest food comes first.

    Example:
    Q: Compare X and Y
    A: X is healtier as Y.
    1) X: 8/10 - Very healthy but high in fructose
    2) Y: 3/10 - High in sugar and fat
    """,
    model="litellm/bedrock/eu.amazon.nova-lite-v1:0",
)

## No Memory

In [3]:
with trace("Compare without Memory") as t:
    result = await Runner.run(nutrition_agent, "Which is healthier, bananas or lollipop?")

print(f"Output: {result.final_output}")
print(f"\nTrace URL: https://platform.openai.com/logs/trace?trace_id={t.trace_id}")


Output: A: Bananas are healthier than lollipops.

1) Bananas: 9/10
   - High in essential nutrients such as potassium, vitamin B6, and dietary fiber.
   - Low in fat and contain natural sugars.
   
2) Lollipops: 1/10
   - High in sugar with little to no nutritional value.
   - Can contribute to tooth decay and provide empty calories.

Trace URL: https://platform.openai.com/logs/trace?trace_id=trace_c316c03f0207465db22dbd7c3e9ccfab


In [4]:
with trace("Add Apple - without Memory") as t:
    result = await Runner.run(nutrition_agent, "Add apples to the comparison")

print(f"Output: {result.final_output}")
print(f"\nTrace URL: https://platform.openai.com/logs/trace?trace_id={t.trace_id}")

Output: Certainly! Let's compare apples with a few other common foods based on their health scores out of 10.

1. **Apples: 9/10**
   - Apples are rich in fiber, vitamins (like vitamin C), and antioxidants. They are relatively low in calories and can help with digestion and heart health.

2. **Blueberries: 8/10**
   - Blueberries are packed with antioxidants, vitamins, and fiber. They are low in calories and have anti-inflammatory properties.

3. **Spinach: 10/10**
   - Spinach is a nutritional powerhouse, loaded with vitamins A, C, and K, as well as minerals like iron and calcium. It is very low in calories and high in fiber.

4. **Salmon: 7/10**
   - Salmon is an excellent source of high-quality protein and omega-3 fatty acids, which are great for heart health. However, it is higher in calories compared to fruits and vegetables.

5. **Whole Grains (like brown rice or quinoa): 8/10**
   - Whole grains provide a good source of fiber, vitamins, and minerals. They are lower in calories t

## Short Term Memory

In [5]:
session = SQLiteSession("conversation_history")

In [6]:
result = await Runner.run(
    nutrition_agent, "Which is healthier, bananas or lollipop?", session=session
)
print(result.final_output)

Here's a comparison of the healthiness of bananas and lollipops:

1) Bananas: 9/10 - Rich in vitamins, minerals, and fiber. They are low in calories and provide natural sugars and potassium, which are beneficial for health.
   
2) Lollipops: 1/10 - High in sugar with little to no nutritional value. They can contribute to cavities and provide empty calories.

Therefore, bananas are significantly healthier than lollipops.


In [7]:
with trace("Compare with Memory") as t:
    result = await Runner.run(
        nutrition_agent, "Add apples to the comparison", session=session
    )

print(f"Output: {result.final_output}")
print(f"\nTrace URL: https://platform.openai.com/logs/trace?trace_id={t.trace_id}")

Output: Here’s an updated comparison of the healthiness of bananas, apples, and lollipops:

1) Bananas: 9/10 - Rich in vitamins, minerals, and fiber. They are low in calories and provide natural sugars and potassium, which are beneficial for health.

2) Apples: 8/10 - High in fiber, vitamin C, and various antioxidants. They are low in calories and have been associated with numerous health benefits, including improved heart health and reduced risk of chronic diseases.

3) Lollipops: 1/10 - High in sugar with little to no nutritional value. They can contribute to cavities and provide empty calories.

In order of healthiness, from healthiest to least healthy:
1) Bananas
2) Apples
3) Lollipops

Trace URL: https://platform.openai.com/logs/trace?trace_id=trace_f86eb44d978a4eba888e319892342fd4
